## Tools

Models can request to call tools that perform tasks such as fetching data, performing calculations, or interacting with external APIs. These tools can be integrated into the model's workflow to enhance its capabilities and provide more accurate and relevant responses.

Tools are pairing of:
 - A Schema, including the tool's name, description, and input/output specifications (often in JSON Schema).
 - An implementation, which is the actual code that executes the tool's functionality based on the provided input and returns the output in the specified format.

In [1]:
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrots talk?")
response

AIMessage(content='Parrots "talk" because they have evolved to be highly social animals that rely heavily on **vocal communication** to bond with their flock, establish territory, and signal danger. In the wild, parrots do not mimic human speech; instead, they mimic the sounds of their environment, including other birds, predators, and eventually, humans.\n\nHere’s a breakdown of why and how parrots learn to mimic human speech:\n\n### 1. Social Bonding\nParrots are flock animals. In the wild, they constantly chatter, call, and make noise to maintain social cohesion. When a parrot is kept as a pet, it views its human owner as part of its flock. Mimicking human speech is a way for the parrot to:\n- **Integrate** with its "flock" (the humans).\n- **Seek attention** or interaction.\n- **Express emotions** like excitement, frustration, or joy.\n\n### 2. Innate Ability to Mimic\nParrots, especially species like African Greys, Amazons, Cockatoos, and Budgerigars, have a unique anatomical and 

In [19]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    return f"The current weather in {location} is sunny with a temperature of 25°C."

def get_temperature(location: str) -> str:
    """Get the current temperature for a given location."""
    return f"The current temperature in {location} is 25°C."


model_with_tools = model.bind_tools([get_weather, get_temperature])

In [22]:
response = model_with_tools.invoke("What is the weather like in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'tool_calls': [{'id': 'p5a6wsvgm', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 27, 'prompt_tokens': 340, 'total_tokens': 367, 'completion_time': 0.074419052, 'completion_tokens_details': None, 'prompt_time': 0.02514466, 'prompt_tokens_details': None, 'queue_time': 0.268539021, 'total_time': 0.099563712}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a20219-2495-7238-92a8-75582e98e9db-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'p5a6wsvgm', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 340, 'output_tokens': 27, 'total_tokens': 367}
Tool: get_weather
Args: {'location': 'New York'}


### Tool Execution Loops

In [23]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What is the weather like in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute the tool calls and get the results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Pass result back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's currently sunny in New York with a temperature of 25°C (77°F).
